<a href="https://colab.research.google.com/github/Raajarapu/VIDEO_SUM/blob/main/Videos_Summarization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
!pip install moviepy transformers torch librosa
!pip install git+https://github.com/openai/whisper.git


  Cloning https://github.com/openai/whisper.git to /tmp/pip-req-build-ws1tdgz7
  Running command git clone --filter=blob:none --quiet https://github.com/openai/whisper.git /tmp/pip-req-build-ws1tdgz7
  Resolved https://github.com/openai/whisper.git to commit c0d2f624c09dc18e709e37c2ad90c039a4eb72a2
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [12]:
!pip install moviepy git+https://github.com/openai/whisper.git transformers torch

  Cloning https://github.com/openai/whisper.git to /tmp/pip-req-build-zmrrbvd9
  Running command git clone --filter=blob:none --quiet https://github.com/openai/whisper.git /tmp/pip-req-build-zmrrbvd9
  Resolved https://github.com/openai/whisper.git to commit c0d2f624c09dc18e709e37c2ad90c039a4eb72a2
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


# 1st video summarization


In [15]:
!pip install transformers sentencepiece torch torchvision pillow moviepy opencv-python-headless

import os
import cv2
from moviepy.editor import VideoFileClip
from transformers import BlipProcessor, BlipForConditionalGeneration, pipeline
from tqdm import tqdm

# Step 1: Extract frames
video_path = "/content/4000-360.mp4"
frames_dir = "/content/frames_scene"
os.makedirs(frames_dir, exist_ok=True)

clip = VideoFileClip(video_path)
fps = int(clip.fps)
duration = int(clip.duration)
interval = max(1, duration // 6)  # extract around 6 frames
print("[INFO] Extracting frames...")
for i, t in enumerate(range(0, duration, interval)):
    frame = clip.get_frame(t)
    frame_path = f"{frames_dir}/frame_{i}.jpg"
    cv2.imwrite(frame_path, cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))
clip.close()
print("[INFO] Frames extracted successfully.")

# Step 2: Use BLIP (Bootstrapped Language-Image Pretraining) for image captioning
from PIL import Image
import torch

processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base").to("cpu")

captions = []
print("[INFO] Generating scene captions...")
for file in tqdm(sorted(os.listdir(frames_dir))):
    if file.endswith(".jpg"):
        image = Image.open(os.path.join(frames_dir, file)).convert("RGB")
        inputs = processor(image, return_tensors="pt").to("cpu")
        out = model.generate(**inputs)
        caption = processor.decode(out[0], skip_special_tokens=True)
        captions.append(caption)

print("\n📝 Scene Captions from Frames:")
for i, c in enumerate(captions, 1):
    print(f"{i}. {c}")

# Step 3: Summarize all captions into a short paragraph
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

joined_captions = " ".join(captions)
summary = summarizer(joined_captions, max_length=120, min_length=50, do_sample=False)
summary_text = summary[0]['summary_text']

print("\n✅ FINAL VIDEO SUMMARY (3-4 lines):")
print(summary_text)


[INFO] Extracting frames...


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


[INFO] Frames extracted successfully.


preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

...

[INFO] Generating scene captions...



100%|██████████| 7/7 [01:16<00:00, 10.87s/it]



📝 Scene Captions from Frames:
1. a busy intersection with many cars and people
2. a group of people are walking across the street
3. a busy intersection with cars and people walking around
4. a group of people are walking across a street
5. a group of people crossing a street in a city
6. a busy intersection with cars and people crossing the street
7. a busy intersection with many yellow taxis and people


Device set to use cpu
Your max_length is set to 120, but your input_length is only 66. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=33)



✅ FINAL VIDEO SUMMARY (3-4 lines):
A group of people are walking across a street in a city. A busy intersection with many cars and people. a busy intersection. with many yellow taxis and people a group of. people are walked across the street. a group. of people crossing a street a busy. intersection with people and cars.


# 2nd Video Summarization

In [17]:
# 1️⃣ Install dependencies
!pip install -q transformers pillow torchvision torch moviepy

# 2️⃣ Imports
from moviepy.editor import VideoFileClip
from transformers import BlipProcessor, BlipForConditionalGeneration, pipeline
from PIL import Image
import torch

# 3️⃣ Define paths
video_path = "/content/4547-360.mp4"

# 4️⃣ Extract key frames
print("[INFO] Extracting key frames from silent video...")
clip = VideoFileClip(video_path)
frames = []
for t in range(0, int(clip.duration), 3):  # one frame every 3 seconds
    frame = clip.get_frame(t)
    frames.append(Image.fromarray(frame))
clip.close()
print(f"[INFO] Extracted {len(frames)} frames ✅")

# 5️⃣ Load BLIP model for visual captioning
print("[INFO] Generating captions from frames...")
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base").to("cuda" if torch.cuda.is_available() else "cpu")

captions = []
for i, frame in enumerate(frames):
    inputs = processor(images=frame, return_tensors="pt").to(model.device)
    out = model.generate(**inputs)
    caption = processor.decode(out[0], skip_special_tokens=True)
    captions.append(caption)

scene_text = " ".join(captions)
print("\n🖼️ VISUAL CAPTIONS SAMPLE:\n", scene_text[:500] + "...")

# 6️⃣ Summarize the combined captions
print("\n[INFO] Summarizing scene description...")
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")
summary = summarizer(scene_text, max_length=120, min_length=40, do_sample=False)[0]['summary_text']

print("\n✅ FINAL VISUAL SUMMARY (3–4 lines):")
print(summary)


[INFO] Extracting key frames from silent video...
[INFO] Extracted 10 frames ✅
[INFO] Generating captions from frames...

🖼️ VISUAL CAPTIONS SAMPLE:
 two business people having a meeting in a modern office a group of people sitting around a table a man and woman sitting at a table a woman sitting at a table with two men a man and woman sitting at a table a man and woman sitting at a table with a laptop a man and woman sitting at a table a group of people sitting around a table a man and woman sitting at a table two people sitting at a table with their hands up...

[INFO] Summarizing scene description...


Device set to use cpu
Your max_length is set to 120, but your input_length is only 90. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=45)



✅ FINAL VISUAL SUMMARY (3–4 lines):
Two business people having a meeting in a modern office. A group of people sitting around a table. A man and woman sitting at a table with a laptop. Two people with their hands up.


In [20]:
!pip install -q transformers torch torchvision decord pillow tqdm

from transformers import BlipProcessor, BlipForConditionalGeneration, VideoMAEFeatureExtractor, VideoMAEForVideoClassification, pipeline
from decord import VideoReader, cpu
from PIL import Image
import torch, numpy as np
from tqdm import tqdm

# ---------- 1️⃣ Load Video ----------
video_path = "/content/4547-360.mp4"

# ---------- 2️⃣ Extract Frames ----------
print("[INFO] Extracting frames...")
vr = VideoReader(video_path, ctx=cpu(0))
total_frames = len(vr)
indices = np.linspace(0, total_frames-1, num=12, dtype=int)
frames = vr.get_batch(indices).asnumpy()
print(f"[INFO] Extracted {len(frames)} frames ✅")

# ---------- 3️⃣ Generate Visual Captions ----------
print("[INFO] Generating visual captions...")
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")

captions = []
for f in tqdm(frames):
    image = Image.fromarray(f)
    inputs = processor(image, return_tensors="pt")
    out = model.generate(**inputs, max_new_tokens=30)
    caption = processor.decode(out[0], skip_special_tokens=True)
    captions.append(caption)

print("\n🖼️ VISUAL CAPTIONS SAMPLE:")
print(", ".join(captions))

# ---------- 4️⃣ Recognize Actions ----------
print("\n[INFO] Loading action recognition model...")
feat_extractor = VideoMAEFeatureExtractor.from_pretrained("MCG-NJU/videomae-base-finetuned-kinetics")
act_model = VideoMAEForVideoClassification.from_pretrained("MCG-NJU/videomae-base-finetuned-kinetics")

# Prepare sampled frames for model
sample_indices = np.linspace(0, total_frames-1, num=16, dtype=int)
clip = vr.get_batch(sample_indices).asnumpy()
inputs = feat_extractor(list(clip), return_tensors="pt")

print("[INFO] Recognizing actions...")
with torch.no_grad():
    outputs = act_model(**inputs)
pred = torch.nn.functional.softmax(outputs.logits, dim=1)
label = act_model.config.id2label[pred.argmax().item()]

print(f"🎯 Action detected: {label}")

# ---------- 5️⃣ Summarize Everything ----------
print("\n[INFO] Summarizing the scene...")
summarizer = pipeline("summarization", model="facebook/bart-large-cnn", device="cpu")

context = " ".join(captions) + f" People are performing action: {label}."
summary = summarizer(context, max_length=120, min_length=40, do_sample=False)[0]['summary_text']

print("\n✅ FINAL VISUAL SUMMARY:")
print(summary)


[INFO] Extracting frames...
[INFO] Extracted 12 frames ✅
[INFO] Generating visual captions...


100%|██████████| 12/12 [00:42<00:00,  3.50s/it]



🖼️ VISUAL CAPTIONS SAMPLE:
two business people having a meeting in a modern office, two people sitting at a table in an office, a man and woman sitting at a table, a woman and a man sitting at a table, two people sitting at a table in an office, a man and woman sitting at a table with a laptop, a man and woman sitting at a table, a man and woman sitting at a table in an office, two people sitting at a table in an office, a man and woman sitting at a table, a man and woman sitting at a table, two women sitting at a table with their hands up

[INFO] Loading action recognition model...
[INFO] Recognizing actions...
🎯 Action detected: playing monopoly

[INFO] Summarizing the scene...


Device set to use cpu



✅ FINAL VISUAL SUMMARY:
Two business people having a meeting in a modern office. Two women sitting at a table with their hands up. People are performing action: playing monopoly. Two people sitting at table in an office.


# 3rd Video Summarization

In [21]:
!pip install -q transformers torch torchvision pillow decord ultralytics tqdm

from transformers import BlipProcessor, BlipForConditionalGeneration, pipeline
from ultralytics import YOLO
from decord import VideoReader, cpu
from PIL import Image
import numpy as np
import torch
from tqdm import tqdm

# ------------------ 1️⃣ Load video ------------------
video_path = "/content/51238-360.mp4"
print("[INFO] Loading video...")

vr = VideoReader(video_path, ctx=cpu(0))
total_frames = len(vr)
indices = np.linspace(0, total_frames-1, num=12, dtype=int)
frames = vr.get_batch(indices).asnumpy()
print(f"[INFO] Extracted {len(frames)} key frames ✅")

# ------------------ 2️⃣ Object Detection ------------------
print("[INFO] Detecting objects on table...")
model_yolo = YOLO("yolov8m.pt")  # medium model for better accuracy
detected_objects = []

for f in tqdm(frames):
    results = model_yolo.predict(f, verbose=False)
    objs = set()
    for box in results[0].boxes:
        cls = int(box.cls)
        objs.add(results[0].names[cls])
    detected_objects.extend(list(objs))

unique_objects = list(set(detected_objects))
print(f"🧂 Objects detected: {unique_objects}")

# ------------------ 3️⃣ Generate visual captions ------------------
print("[INFO] Generating detailed captions...")
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")

captions = []
for f in tqdm(frames):
    image = Image.fromarray(f)
    inputs = processor(image, return_tensors="pt")
    out = model.generate(**inputs, max_new_tokens=40)
    caption = processor.decode(out[0], skip_special_tokens=True)
    captions.append(caption)

print("\n🖼️ SAMPLE VISUAL CAPTIONS:")
print(", ".join(captions))

# ------------------ 4️⃣ Create chunked description ------------------
scene_chunks = []
for i, cap in enumerate(captions):
    chunk = f"Scene {i+1}: {cap}"
    scene_chunks.append(chunk)

scene_text = " ".join(scene_chunks)
context = f"Detected objects on table include: {', '.join(unique_objects)}. Visual description: {scene_text}"

# ------------------ 5️⃣ Summarize the text ------------------
print("\n[INFO] Generating summary from visual text...")
summarizer = pipeline("summarization", model="facebook/bart-large-cnn", device="cpu")

summary = summarizer(context, max_length=150, min_length=60, do_sample=False)[0]['summary_text']

print("\n✅ FINAL VISUAL SUMMARY:")
print(summary)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 19.5 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
[INFO] Loading video...
[INFO] Extracted 12 key frames ✅
[INFO] Detecting objects on table...


100%|██████████| 12/12 [00:13<00:00,  1.09s/it]


🧂 Objects detected: ['person', 'dining table', 'cup', 'knife', 'cell phone', 'spoon', 'bowl']
[INFO] Generating detailed captions...


100%|██████████| 12/12 [00:40<00:00,  3.41s/it]



🖼️ SAMPLE VISUAL CAPTIONS:
two women sitting at a table eating food, two women sitting at a table eating food, two women sitting at a table eating food, two women sitting at a table eating food, two women sitting at a table eating food, two women sitting at a table eating su, two women sitting at a table eating su, two women sitting at a table eating food, two women sitting at a table eating food, two women eating suki at a table, two women eating suki suki at a table, two women sitting at a table eating food

[INFO] Generating summary from visual text...


Device set to use cpu



✅ FINAL VISUAL SUMMARY:
Scene 1: Two women sitting at a table eating food. Scene 2: two women eating suki suki at atable. Scene 3: two woman eating food at table. Scene 4: two people sitting at table eatingFood. Scene 5: Two people sitting on table eating Food. Scene 6: People eating food on table.


# 4th Video Summarization

In [2]:
# ================================================================
# FINAL FIXED VERSION - VIDEO -> TEXT -> MULTI-ALGO SUMMARIZATION
# ================================================================

!pip install transformers==4.44.2 torchvision opencv-python Pillow torch tqdm sentencepiece

import cv2
import os
import torch
import numpy as np
from PIL import Image
from transformers import BlipProcessor, BlipForConditionalGeneration
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, BartForConditionalGeneration, BartTokenizer, pipeline
from tqdm import tqdm

# ================================================================
# STEP 1: Setup Paths
# ================================================================
video_path = "/content/mixkit-two-thieves-recorded-on-a-security-camera-31372-hd-ready.mp4"
frames_dir = "/content/keyframes"
os.makedirs(frames_dir, exist_ok=True)

# ================================================================
# STEP 2: Extract Key Frames
# ================================================================
print("[INFO] Extracting key frames...")
cap = cv2.VideoCapture(video_path)
frame_rate = int(cap.get(cv2.CAP_PROP_FPS))
count, saved = 0, 0
frames = []

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    if count % (frame_rate * 2) == 0:  # every 2 seconds
        frame_path = os.path.join(frames_dir, f"frame_{saved}.jpg")
        cv2.imwrite(frame_path, frame)
        frames.append(frame_path)
        saved += 1
    count += 1

cap.release()
print(f"[INFO] Extracted {len(frames)} frames.")

# ================================================================
# STEP 3: Generate Scene Descriptions with BLIP
# ================================================================
print("[INFO] Generating scene descriptions...\n")

device = "cuda" if torch.cuda.is_available() else "cpu"
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base").to(device)

scene_descriptions = []

for frame_path in tqdm(frames, desc="Captioning Frames"):
    raw_image = Image.open(frame_path).convert('RGB')
    inputs = processor(raw_image, return_tensors="pt").to(device)
    out = model.generate(**inputs, max_new_tokens=30)
    caption = processor.decode(out[0], skip_special_tokens=True)
    scene_descriptions.append(caption)

print("\n[INFO] Scene Descriptions Generated:\n")
for i, caption in enumerate(scene_descriptions, 1):
    print(f"Frame {i}: {caption}")

video_text = " ".join(scene_descriptions)

# ================================================================
# STEP 4: Summarization (3 Different Models)
# ================================================================
print("\n[INFO] Summarizing using multiple algorithms...\n")

# --- Algorithm 1: T5-Small ---
model_t5 = AutoModelForSeq2SeqLM.from_pretrained("t5-small")
tokenizer_t5 = AutoTokenizer.from_pretrained("t5-small")
inputs = tokenizer_t5("summarize: " + video_text, return_tensors="pt", max_length=512, truncation=True)
summary_ids = model_t5.generate(inputs["input_ids"], max_new_tokens=80, min_length=20, num_beams=4)
summary_t5 = tokenizer_t5.decode(summary_ids[0], skip_special_tokens=True)

# --- Algorithm 2: BART ---
model_bart = BartForConditionalGeneration.from_pretrained("facebook/bart-large-cnn")
tokenizer_bart = BartTokenizer.from_pretrained("facebook/bart-large-cnn")
inputs_bart = tokenizer_bart([video_text], max_length=1024, return_tensors="pt", truncation=True)
summary_ids_bart = model_bart.generate(inputs_bart["input_ids"], max_new_tokens=80, min_length=20, num_beams=4)
summary_bart = tokenizer_bart.decode(summary_ids_bart[0], skip_special_tokens=True)

# --- Algorithm 3: DistilGPT2 (Creative Summary, FIXED) ---
summarizer_gpt = pipeline("text-generation", model="distilgpt2")
summary_gpt = summarizer_gpt(
    "Summarize this CCTV scene briefly: " + video_text,
    max_new_tokens=60,     # ✅ fixed
    truncation=True,       # ✅ fixed
    num_return_sequences=1
)[0]['generated_text']

# ================================================================
# STEP 5: Structured Output
# ================================================================
print("\n🧩 STRUCTURED OUTPUT\n")
print("📸 SCENE DESCRIPTIONS (FRAME-WISE):\n")
for i, caption in enumerate(scene_descriptions, 1):
    print(f"{i}. {caption}")

print("\n🧠 SUMMARIZATION RESULTS:\n")
print("🔹 T5 SUMMARY:\n", summary_t5)
print("\n🔹 BART SUMMARY:\n", summary_bart)
print("\n🔹 GPT2 SUMMARY (Creative):\n", summary_gpt)


[INFO] Extracting key frames...
[INFO] Extracted 11 frames.
[INFO] Generating scene descriptions...



Captioning Frames: 100%|██████████| 11/11 [01:18<00:00,  7.13s/it]



[INFO] Scene Descriptions Generated:

Frame 1: a black and white photo of a living room
Frame 2: a camera view of a room with a desk and a television
Frame 3: a room with a couch and a television
Frame 4: a room with a couch, television and a table
Frame 5: a man sitting on a couch in a living room
Frame 6: a room with a couch and a television
Frame 7: a man in a room with a camera
Frame 8: a camera view of a living room
Frame 9: a man in a black suit is standing on a table
Frame 10: a room with a couch and a table
Frame 11: a person is standing in a room with a camera

[INFO] Summarizing using multiple algorithms...



Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



🧩 STRUCTURED OUTPUT

📸 SCENE DESCRIPTIONS (FRAME-WISE):

1. a black and white photo of a living room
2. a camera view of a room with a desk and a television
3. a room with a couch and a television
4. a room with a couch, television and a table
5. a man sitting on a couch in a living room
6. a room with a couch and a television
7. a man in a room with a camera
8. a camera view of a living room
9. a man in a black suit is standing on a table
10. a room with a couch and a table
11. a person is standing in a room with a camera

🧠 SUMMARIZATION RESULTS:

🔹 T5 SUMMARY:
 black and white photo of a living room a camera view of a room with a desk and a television a room with a couch and a television a man sitting on a couch in a living room a man in a black suit standing on a table a man sitting on a couch a room with a couch and a

🔹 BART SUMMARY:
 a black and white photo of a living room. a camera view of a room with a desk and a television. a man in a black suit is standing on a table.

🔹 G